In [1]:

import numpy as np
import matplotlib.pyplot as plt
import mplhep
import sys
from sklearn.decomposition import PCA
from typing import List, Optional
import timeit
import awkward as ak
import torch
import torch.nn as nn
from torch.nn import Parameter 
from torch.nn.init import xavier_uniform_, xavier_normal_, constant_
import torch
from torch import nn, Tensor
from typing import Optional
import torch.nn.functional as F
from typing import Optional, Tuple
_is_fastpath_enabled: bool = True
from torch.overrides import (
    handle_torch_function,
    has_torch_function,
    has_torch_function_unary,
    has_torch_function_variadic,
)
linear = torch._C._nn.linear
import math
import random
import warnings
import copy
from torch._C import _add_docstr, _infer_size
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colorbar import ColorbarBase
from matplotlib.cm import ScalarMappable

from functools import partial
from weaver.utils.logger import _logger
import os
import uproot
from torch.utils.data import IterableDataset, DataLoader
import numpy as np
from tqdm import tqdm
from torch._torch_docs import reproducibility_notes, sparse_support_notes, tf32_notes

import model_utils as mu

In [2]:
model_type = 'n8_k4_ffn1_cap2_auxfree_10_pct.pt'
n_experts = 8
k_experts = 4
ffn_ratio = 1
capacity_factor = 2

model = mu.get_moe_model('jc_full', moe_num_experts=n_experts, moe_top_k=k_experts, ffn_ratio=ffn_ratio, moe_capacity_factor=capacity_factor)[0]
model.load_state_dict(torch.load('./models/' + model_type, map_location=torch.device('cpu')))

<All keys matched successfully>

In [3]:
features, labels, mask, points, vectors = mu.load_jet_data(stop=100, data_dir='./jc_full_data')

In [4]:
router_hook = mu.Router_Hook(model)
model.eval()
with torch.no_grad():
    _ = model(torch.from_numpy(points),torch.from_numpy(features),
                                torch.from_numpy(vectors),torch.from_numpy(mask))

Registered hook onto particle module


/home/tim_legge/MoE_Interpretability/model_utils.py:2878: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=self.use_amp):
/home/tim_legge/MoE_Interpretability/model_utils.py:576: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


In [5]:
flat_features = features.transpose(0,2,1) # (N, C, P) -> (N, P, C)
# features: (N, P, C) -> (N*P, C)
flat_features = flat_features.reshape(-1, features.shape[1])
flat_features = flat_features[router_hook.valid_indices,:]

pt = flat_features[:,0]
energy = flat_features[:,1]
pt_rel = flat_features[:,2]
e_rel = flat_features[:,3]
delta_R = flat_features[:,4]
charge = flat_features[:,5]
pid = flat_features[:,6:11]
pid = np.argmax(pid, axis=1) # 0: charged_hadron, 1: neutral_hadron, 2: photon, 3: electron, 4: muon

print(pt.shape, energy.shape, pt_rel.shape, e_rel.shape, delta_R.shape, charge.shape, pid.shape)

(4130,) (4130,) (4130,) (4130,) (4130,) (4130,) (4130,)


In [6]:
pt_q1 = np.percentile(pt, 25)
pt_q2 = np.percentile(pt, 50)
pt_q3 = np.percentile(pt, 75)

energy_q1 = np.percentile(energy, 25)
energy_q2 = np.percentile(energy, 50)
energy_q3 = np.percentile(energy, 75)

pt_rel_q1 = np.percentile(pt_rel, 25)
pt_rel_q2 = np.percentile(pt_rel, 50)
pt_rel_q3 = np.percentile(pt_rel, 75)

e_rel_q1 = np.percentile(e_rel, 25)
e_rel_q2 = np.percentile(e_rel, 50)
e_rel_q3 = np.percentile(e_rel, 75)

delta_R_q1 = np.percentile(delta_R, 25)
delta_R_q2 = np.percentile(delta_R, 50)
delta_R_q3 = np.percentile(delta_R, 75)

In [7]:
particle_q_ids = np.zeros((len(pt), 4))
for particle in range(len(pt)):
    # determine which quartiles each particle belongs to
    pt_quartile = 1 if pt[particle] < pt_q1 else (2 if pt[particle] < pt_q2 else (3 if pt[particle] < pt_q3 else 4))
    energy_quartile = 1 if energy[particle] < energy_q1 else (2 if energy[particle] < energy_q2 else (3 if energy[particle] < energy_q3 else 4))
    pt_rel_quartile = 1 if pt_rel[particle] < pt_rel_q1 else (2 if pt_rel[particle] < pt_rel_q2 else (3 if pt_rel[particle] < pt_rel_q3 else 4))
    e_rel_quartile = 1 if e_rel[particle] < e_rel_q1 else (2 if e_rel[particle] < e_rel_q2 else (3 if e_rel[particle] < e_rel_q3 else 4))
    delta_R_quartile = 1 if delta_R[particle] < delta_R_q1 else (2 if delta_R[particle] < delta_R_q2 else (3 if delta_R[particle] < delta_R_q3 else 4))
    charge_val = charge[particle]
    pid_val = pid[particle]
    #    particle_q_ids[particle] = [pt_quartile, energy_quartile, pt_rel_quartile, e_rel_quartile, delta_R_quartile, charge_id]
    particle_q_ids[particle] = [pt_quartile, delta_R_quartile, charge_val, pid_val]

In [8]:
print(particle_q_ids[:10])

[[ 4.  2.  1.  3.]
 [ 4.  2.  0.  2.]
 [ 4.  1.  0.  2.]
 [ 4.  2.  0.  1.]
 [ 4.  1.  0.  1.]
 [ 4.  2. -1.  0.]
 [ 4.  2. -1.  0.]
 [ 4.  2.  1.  0.]
 [ 4.  1.  0.  2.]
 [ 4.  1.  1.  0.]]


In [9]:
weights = router_hook.expert_weights
assignments = router_hook.expert_assignments
print(weights.shape)
print(weights)
print(assignments.shape)

torch.Size([4130, 8])
tensor([[0.0000, 0.9400, 0.0237,  ..., 0.0000, 0.0000, 0.0178],
        [0.0000, 0.0436, 0.0107,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0681, 0.0177,  ..., 0.0130, 0.0000, 0.0000],
        ...,
        [0.3121, 0.0000, 0.0000,  ..., 0.5437, 0.0243, 0.0000],
        [0.5648, 0.0000, 0.0000,  ..., 0.3050, 0.0752, 0.0000],
        [0.6769, 0.0000, 0.0000,  ..., 0.2275, 0.0412, 0.0000]],
       dtype=torch.float64)
torch.Size([4130, 8])


In [12]:
id_ranges = np.zeros(particle_q_ids.shape[-1], dtype=object)
for feature in range(particle_q_ids.shape[-1]):
    # make the feature values start from 0 to function as indices
    id_min = int(particle_q_ids[:, feature].min())
    id_max = int(particle_q_ids[:, feature].max())
    particle_q_ids[:, feature] -= id_min
    id_ranges[feature] = id_max - id_min + 1

id_ranges = id_ranges.tolist()
id_ranges.append(n_experts)
id_ranges = tuple(id_ranges)
print(id_ranges)
particle_partition = np.zeros(id_ranges)
particle_partition

(4, 4, 3, 5, 8)


array([[[[[0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.]],

         [[0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.]],

         [[0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.]]],


        [[[0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.]],

         [[0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
      

In [53]:
particle_partition[0,0,0,0]=[1,1]

In [17]:
lowest_entropy = 1e10
largest_expert = 0
stats_shape = tuple(list(id_ranges[:-1]) + [3])
perm_stats = np.zeros((stats_shape))
for perm, _ in np.ndenumerate(particle_partition[:-1]):
    perm_slice = np.where(np.all(particle_q_ids == perm[:-1], axis=1))[0]
    perm_weights = weights[perm_slice].numpy()
    perm_assignments = assignments[perm_slice].numpy()
    if len(perm_weights) <= 100:  # skip permutations with too few particles
        continue
    print(f"Permutation: {perm[:-1]}, Number of particles: {len(perm_weights)}")
    assignment_dist = np.sum(perm_assignments, axis=0) / np.sum(perm_assignments)
    particle_partition[perm[:-1]] = assignment_dist
    dist_entropy = -np.sum(assignment_dist * np.log(assignment_dist + 1e-10))
    max_expert = np.max(assignment_dist)
    perm_stats[perm[:-1]] = [dist_entropy, max_expert, len(perm_weights)]
    if dist_entropy < lowest_entropy:
        lowest_entropy = dist_entropy
        best_ent_perm = perm[:-1]
    if max_expert > largest_expert:
        largest_expert = max_expert
        best_expert_perm = perm[:-1]

Permutation: (0, 0, 1, 2), Number of particles: 124
Permutation: (0, 0, 1, 2), Number of particles: 124
Permutation: (0, 0, 1, 2), Number of particles: 124
Permutation: (0, 0, 1, 2), Number of particles: 124
Permutation: (0, 0, 1, 2), Number of particles: 124
Permutation: (0, 0, 1, 2), Number of particles: 124
Permutation: (0, 0, 1, 2), Number of particles: 124
Permutation: (0, 0, 1, 2), Number of particles: 124
Permutation: (0, 1, 1, 2), Number of particles: 136
Permutation: (0, 1, 1, 2), Number of particles: 136
Permutation: (0, 1, 1, 2), Number of particles: 136
Permutation: (0, 1, 1, 2), Number of particles: 136
Permutation: (0, 1, 1, 2), Number of particles: 136
Permutation: (0, 1, 1, 2), Number of particles: 136
Permutation: (0, 1, 1, 2), Number of particles: 136
Permutation: (0, 1, 1, 2), Number of particles: 136
Permutation: (0, 2, 1, 2), Number of particles: 183
Permutation: (0, 2, 1, 2), Number of particles: 183
Permutation: (0, 2, 1, 2), Number of particles: 183
Permutation:

In [16]:
print(lowest_entropy, best_ent_perm)
print(largest_expert, best_expert_perm)

1.6269689 (0, 2, 1, 2)
0.25 (0, 0, 1, 2)


In [ ]:
# entropy and largest expert distribution across permutations
entropies = particle_partition[..., 0].flatten()
print(entropies)

[1.94787359 0.         0.         1.38629436 1.38629436 0.
 1.38629436 1.69283795 0.         0.         1.85216212 0.
 0.         1.66094756 0.         1.95388794 0.         0.
 0.         0.         0.         1.54542303 1.64096999 0.
 0.         1.98419118 0.         0.         0.         0.
 1.96601737 0.         0.         1.55958116 0.         0.
 1.60318518 1.62696886 0.         0.         1.94145918 0.
 0.         0.         0.         1.95370805 0.         0.
 1.38629436 0.         0.         1.70717442 1.64339411 0.
 0.         1.9657464  0.         0.         1.38629436 1.55958116
 1.94416058 0.         0.         0.         0.         0.
 1.69949353 1.77610612 0.         0.         1.94582641 0.
 0.         0.         0.         1.90722466 0.         0.
 1.55958116 0.         0.         1.55803478 1.76044273 0.
 0.         2.02241755 0.         0.         0.         1.73286796
 2.03401065 0.         0.         1.38629436 0.         0.
 1.72710896 1.79802251 0.         0.    